# HW2 Image Classification

Submit this Jupyter notebook with the code outputs, along with the report.
Ensure that the best accuracy score achieved is mentioned in the report.

# Check GPU Type

In [1]:
!nvidia-smi

Thu Sep 17 16:09:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
def get_device():
    ''' Get device (if GPU is available, use GPU) '''
    return 'cuda' if torch.cuda.is_available() else 'cpu'

import torch
print(get_device())
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda
True
Tesla T4


# Get Data
Notes: if the links are dead, you can download the data directly from Kaggle and upload it to the workspace, or you can use the Kaggle API to directly download the data into colab.


In [ ]:
# Download Link
# Link 1 (Dropbox): https://www.dropbox.com/s/up5q1gthsz3v0dq/food-11.zip?dl=0
# Link 2 (Google Drive): https://drive.google.com/file/d/1tbGNwk1yGoCBdu4Gi_Cia7EJ9OhubYD9/view?usp=share_link

# (1) dropbox link
# Using curl instead of wget: wget isn't available on this Windows/local setup,
# curl is (native on Windows 10+, Git Bash, and Kaggle), so this works everywhere.
!curl -L -o food11.zip "https://www.dropbox.com/s/up5q1gthsz3v0dq/food-11.zip?dl=1"

# (2) google drive link
# !pip install gdown --upgrade
# !gdown --id '1tbGNwk1yGoCBdu4Gi_Cia7EJ9OhubYD9' --output food11.zip

In [3]:
! unzip food11.zip

Archive:  food11.zip
   creating: valid/
  inflating: valid/9_2898.jpg        
  inflating: valid/9_3053.jpg        
  inflating: valid/9_294.jpg         
  inflating: valid/9_2048.jpg        
  inflating: valid/9_3190.jpg        
  inflating: valid/9_1020.jpg        
  inflating: valid/9_2861.jpg        
  inflating: valid/9_3012.jpg        
  inflating: valid/9_723.jpg         
  inflating: valid/9_2008.jpg        
  inflating: valid/9_2755.jpg        
  inflating: valid/9_3225.jpg        
  inflating: valid/9_976.jpg         
  inflating: valid/9_2319.jpg        
  inflating: valid/9_2961.jpg        
  inflating: valid/9_1706.jpg        
  inflating: valid/9_1504.jpg        
  inflating: valid/9_698.jpg         
  inflating: valid/9_218.jpg         
  inflating: valid/9_749.jpg         
  inflating: valid/2_2594.jpg        
  inflating: valid/2_667.jpg         
  inflating: valid/2_1415.jpg        
  inflating: valid/2_3426.jpg        
  inflating: valid/2_423.jpg         
  inflati

# Import Packages

In [ ]:
# Naming convention: <tier>_<variant>, e.g. simple_v1, medium_aug1, strong_resnet18.
# Keeps checkpoints/logs from colliding across experiments (both the checkpoint
# filename and the t-SNE loading step key off this value).
# exp_10: exp_9's fair holdout comparison showed 3-fold CV + ensembling
# actually UNDERPERFORMED exp_7's single model (0.676 ensemble vs 0.707
# single) - each fold only trained on ~2/3 of ./train, and less data per
# model apparently cost more than 3-way averaging recovered. Dropping
# ensembling for now and going back to a single model, but scaling up:
# ResNet34 (deeper than exp_7's ResNet18) + more training budget (150
# epochs / patience 25, up from 60/15). dropout=0.5, weight_decay=1e-3,
# lr=0.0001 kept identical to exp_7 so architecture+epochs are the only
# intentional changes (though changing both at once means we can't cleanly
# attribute which one drives any result).
# See ROADMAP.md Experiment Log / report2.md.
_exp_name = "exp_10"

In [5]:
# Import necessary packages.
import numpy as np
import pandas as pd
import torch
import os
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
# "ConcatDataset" and "Subset" are possibly useful when doing semi-supervised learning.
from torch.utils.data import ConcatDataset, DataLoader, Subset, Dataset
from torchvision.datasets import DatasetFolder, VisionDataset
# This is for the progress bar.
from tqdm.auto import tqdm
import random

In [6]:
myseed = 6666  # set a random seed for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(myseed)
torch.manual_seed(myseed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(myseed)

# Transforms
Torchvision provides lots of useful utilities for image preprocessing, data *wrapping* as well as data augmentation.

Please refer to PyTorch official website for details about different transforms.

In [ ]:
# Normally, We don't need augmentations in testing and validation.
# All we need here is to resize the PIL image and transform it into Tensor.
test_tfm = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# However, it is also possible to use augmentation in the testing phase.
# You may use train_tfm to produce a variety of images and then test using ensemble methods
train_tfm = transforms.Compose([
    # RandomResizedCrop resizes to a fixed shape (128x128) while also randomly
    # cropping a sub-region first, so it replaces the plain Resize() and adds
    # scale/aspect-ratio augmentation in one step.
    transforms.RandomResizedCrop(128, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(35),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),

    # ToTensor() should be the last one of the transforms.
    transforms.ToTensor(),
])


In [ ]:
# Verify the Q1 requirement concretely: apply train_tfm to the SAME image
# 5+ times and confirm the outputs are visibly different (not just assumed).
import matplotlib.pyplot as plt

sample_path = sorted(os.path.join("./train", x) for x in os.listdir("./train") if x.endswith(".jpg"))[0]
sample_img = Image.open(sample_path)

n_samples = 5
fig, axes = plt.subplots(1, n_samples + 1, figsize=((n_samples + 1) * 2.5, 3))

axes[0].imshow(sample_img)
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(n_samples):
    augmented = train_tfm(sample_img)  # CxHxW tensor in [0,1]
    axes[i + 1].imshow(augmented.permute(1, 2, 0))
    axes[i + 1].set_title(f"train_tfm #{i+1}")
    axes[i + 1].axis("off")

plt.tight_layout()
plt.savefig("q1_augmentation_grid.png", dpi=150)
plt.show()


# Datasets
The data is labelled by the name, so we load images and label while calling '__getitem__'

In [ ]:
class FoodDataset(Dataset):

    def __init__(self,path,tfm=test_tfm,files = None):
        super(FoodDataset).__init__()
        self.path = path
        self.files = sorted([os.path.join(path,x) for x in os.listdir(path) if x.endswith(".jpg")])
        if files != None:
            self.files = files

        self.transform = tfm

    def __len__(self):
        return len(self.files)

    def __getitem__(self,idx):
        fname = self.files[idx]
        im = Image.open(fname)
        im = self.transform(im)

        try:
            # os.path.basename (not fname.split("/")) so this works on both
            # Unix-style paths (Colab/Kaggle) and Windows-style paths (local).
            label = int(os.path.basename(fname).split("_")[0])
        except:
            label = -1 # test has no label

        return im,label

# Model

In [9]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        # torch.nn.MaxPool2d(kernel_size, stride, padding)
        # input dimension [3, 128, 128]
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1),  # [64, 128, 128]
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [64, 64, 64]

            nn.Conv2d(64, 128, 3, 1, 1), # [128, 64, 64]
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [128, 32, 32]

            nn.Conv2d(128, 256, 3, 1, 1), # [256, 32, 32]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),      # [256, 16, 16]

            nn.Conv2d(256, 512, 3, 1, 1), # [512, 16, 16]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 8, 8]

            nn.Conv2d(512, 512, 3, 1, 1), # [512, 8, 8]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2, 0),       # [512, 4, 4]
        )
        self.fc = nn.Sequential(
            nn.Linear(512*4*4, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 11)
        )

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size()[0], -1)
        return self.fc(out)

In [ ]:
# Strong tier: swap in a predefined torchvision architecture instead of the
# from-scratch Classifier above (kept for reference). No pretrained weights
# per the assignment's constraint - weights=None (not weights=False; None is
# the correct current torchvision API for "no pretrained weights").
# ResNet's adaptive average pooling before the final FC layer means it
# handles our 128x128 input natively - only the FC layer's output needs
# resizing to num_classes=11 (it defaults to 1000, for ImageNet).
#
# exp_6: torchvision's resnet18 has no dropout anywhere by default. exp_4/exp_5
# showed a widening train/valid accuracy gap (overfitting, not just epoch-to-
# epoch noise) - added dropout right before the final FC layer to target this
# directly, per dropout_p below.
# exp_7: exp_6's dropout=0.3 barely moved the needle (best acc essentially
# unchanged, gap still widened) - dosage was too weak, not the wrong idea.
# Pushed dropout_p default up to 0.5.
# exp_10: exp_9's cross-validation/ensembling attempt underperformed exp_7's
# single ResNet18 model, so instead trying a deeper architecture on a single
# model - ResNet34 instead of ResNet18 - alongside a longer training budget.
import torchvision.models as models

def build_model(num_classes=11, dropout_p=0.5):
    model = models.resnet34(weights=None)
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(model.fc.in_features, num_classes),
    )
    return model

# Configurations

In [ ]:
# "cuda" only when GPUs are available.
device = "cuda" if torch.cuda.is_available() else "cpu"

# The number of batch size.
batch_size = 64

# The number of training epochs.
# exp_10: bumped from exp_7's 60 to 150 - ResNet34 is deeper and may need
# more epochs to converge than ResNet18 did, and we want to actually test
# whether more training budget helps now that ensembling didn't.
n_epochs = 150

# If no improvement in 'patience' epochs, early stop.
patience = 25

# For the classification task, we use cross-entropy as the measurement of performance.
criterion = nn.CrossEntropyLoss()

model = build_model().to(device)

# For the classification task, we use Adam optimizer, keeping lr/weight_decay
# identical to exp_7 so architecture+epoch count are the only intentional
# changes from exp_7's recipe.
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-3)

# Dataloader

In [ ]:
# exp_10: back to a single model - no fold-splitting. ./train and ./valid
# are used directly as the training and validation sets, same as exp_7.
train_set = FoodDataset("./train", tfm=train_tfm)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
valid_set = FoodDataset("./valid", tfm=test_tfm)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)

# Start Training

In [ ]:
# exp_10: single-model training loop (no fold loop) - back to the same
# structure exp_7 used, just with the new model/epoch budget from
# Configurations above.
model_save_path = f"{_exp_name}_best.ckpt"
log_path = f"{_exp_name}_log.txt"

# Initialize trackers, these are not parameters and should not be changed.
stale = 0
best_acc = -1.0

for epoch in range(n_epochs):

    # ---------- Training ----------
    # Make sure the model is in train mode before training.
    model.train()

    # These are used to record information in training.
    train_loss = []
    train_accs = []

    for batch in tqdm(train_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch

        # Forward the data. (Make sure data and model are on the same device.)
        logits = model(imgs.to(device))

        # Calculate the cross-entropy loss.
        # We don't need to apply softmax before computing cross-entropy as it is done automatically.
        loss = criterion(logits, labels.to(device))

        # Gradients stored in the parameters in the previous step should be cleared out first.
        optimizer.zero_grad()

        # Compute the gradients for parameters.
        loss.backward()

        # Clip the gradient norms for stable training.
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)

        # Update the parameters with computed gradients.
        optimizer.step()

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        train_loss.append(loss.item())
        train_accs.append(acc)

    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)

    # Print the information.
    print(f"[ Train | {epoch + 1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

    # ---------- Validation ----------
    # Make sure the model is in eval mode so that some modules like dropout are disabled and work normally.
    model.eval()

    # These are used to record information in validation.
    valid_loss = []
    valid_accs = []

    # Iterate the validation set by batches.
    for batch in tqdm(valid_loader):

        # A batch consists of image data and corresponding labels.
        imgs, labels = batch

        # We don't need gradient in validation.
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
            logits = model(imgs.to(device))

        # We can still compute the loss (but not the gradient).
        loss = criterion(logits, labels.to(device))

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()

        # Record the loss and accuracy.
        valid_loss.append(loss.item())
        valid_accs.append(acc)

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)

    # Print the information.
    print(f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

    # update logs
    if valid_acc > best_acc:
        log_line = f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f} -> best"
    else:
        log_line = f"[ Valid | {epoch + 1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}"
    print(log_line)
    with open(log_path, "a") as f:
        f.write(log_line + "\n")

    # save models
    if valid_acc > best_acc:
        print(f"Best model found at epoch {epoch}, saving model")
        torch.save(model.state_dict(), model_save_path)
        best_acc = valid_acc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            print(f"No improvement {patience} consecutive epochs, early stopping")
            break

# Dataloader for test

In [ ]:
# Construct test datasets.
# The argument "loader" tells how torchvision reads the data.
test_set = FoodDataset("./test", tfm=test_tfm)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

# Testing and generate prediction CSV

In [ ]:
# exp_10: single-model prediction - load the one best checkpoint and predict
# directly, no ensembling.
model_best = build_model().to(device)
model_best.load_state_dict(torch.load(model_save_path))
model_best.eval()

prediction = []
with torch.no_grad():
    for data, _ in tqdm(test_loader):
        test_pred = model_best(data.to(device))
        test_label = np.argmax(test_pred.cpu().data.numpy(), axis=1)
        prediction += test_label.squeeze().tolist()

# Q1. Augmentation Implementation
## Implement augmentation by finishing train_tfm in the code with image size of your choice.
## Directly copy the following block and paste it on GradeScope after you finish the code
### Your train_tfm must be capable of producing 5+ different results when given an identical image multiple times.
### Your  train_tfm in the report can be different from train_tfm in your training code.


In [ ]:
train_tfm = transforms.Compose([
    # RandomResizedCrop resizes to a fixed shape (height = width = 128) while
    # also randomly cropping a sub-region first, adding scale/aspect-ratio
    # augmentation in the same step.
    transforms.RandomResizedCrop(128, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(35),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
])

# Q2. Visual Representations Implementation
## Visualize the learned visual representations of the CNN model on the validation set by implementing t-SNE (t-distributed Stochastic Neighbor Embedding) on the output of both top & mid layers (You need to submit 2 images).


In [ ]:
import torch
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.cm as cm
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the trained model
# NOTE: model.cnn[:index] below still assumes the old Classifier's flat
# Sequential structure - ResNet has no .cnn attribute (it's
# conv1/bn1/relu/maxpool/layer1-4/avgpool/fc), so this cell needs a rework
# for Phase 5 (Q2) once we're ready to do the t-SNE visualization.
# exp_10: single model now (no folds), loading model_save_path directly.
model = build_model().to(device)
state_dict = torch.load(model_save_path)
model.load_state_dict(state_dict)
model.eval()

print(model)

In [ ]:
# Load the validation set
valid_set = FoodDataset("./valid", tfm=test_tfm)
valid_loader = DataLoader(valid_set, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

# Extract the representations for the specific layer of model
index = ... # You should find out the index of layer which is defined as "top" or 'mid' layer of your model.
features = []
labels = []
for batch in tqdm(valid_loader):
    imgs, lbls = batch
    with torch.no_grad():
        logits = model.cnn[:index](imgs.to(device))
        logits = logits.view(logits.size()[0], -1)
    labels.extend(lbls.cpu().numpy())
    logits = np.squeeze(logits.cpu().numpy())
    features.extend(logits)

features = np.array(features)
colors_per_class = cm.rainbow(np.linspace(0, 1, 11))

# Apply t-SNE to the features
features_tsne = TSNE(n_components=2, init='pca', random_state=42).fit_transform(features)

# Plot the t-SNE visualization
plt.figure(figsize=(10, 8))
for label in np.unique(labels):
    plt.scatter(features_tsne[labels == label, 0], features_tsne[labels == label, 1], label=label, s=5)
plt.legend()
plt.show()

plt.figure(figsize=(10, 8))
labels = np.array(labels)
target_label = ... # You should put the specific class ID that you want to plot
plt.scatter(features_tsne[labels == target_label, 0], features_tsne[labels == target_label, 1], label=target_label, s=5)
plt.legend()
plt.show()